# 均值方差模型

均值方差模型来源于上世纪 50 年代的 Markowitz 均值方差理论，即在给定约束条件下期望效用最大化。

> **前置阅读**：组合优化模块的整体架构、优化目标与约束条件的基类设计、CVXPC 构造器的使用方法，请先参阅 **[基本框架](基本框架.ipynb)**。

## 数学形式

均值方差模型导出的优化问题的一般形式如下：

$$
\begin{aligned}
  & \underset{\mathbf{w}}{\mathop{\max }}\,\left\{\gamma\cdot\mathbf{\mu}^T\cdot(\mathbf{w}-\mathbf{w}_b)-\frac{\lambda }{2}(\mathbf{w}-\mathbf{w}_b)^T\mathbf{\Sigma}(\mathbf{w}-\mathbf{w}_b)-\operatorname{TC}\left( \mathbf{w} \right) \right\} \\ 
 & \operatorname{TC}\left( \mathbf{w} \right)={\lambda_1}\sum\limits_{i=1}^n{\left| {{w}_{i}}-{{w}_{0i}} \right|} + {\lambda_2}\sum\limits_{i=1}^n{\left( {{w}_{i}}-{{w}_{0i}} \right)^{+}} + {\lambda_3}\sum\limits_{i=1}^n{{\left( {{w}_{i}}-{{w}_{0i}} \right)}^{-}} \\
 & s.t.\ \mathbf{w}\in\mathfrak{C}
\end{aligned}
$$

其中：
* $\mathbf{w}$ 是组合权重，优化变量
* $\mathbf{\mu}$ 是预期收益率
* $\mathbf{\Sigma}$ 是收益率的预期协方差矩阵
* $\gamma$ 是收益项系数（`ExpectedReturnCoef`）
* $\lambda\ge 0$ 是风险厌恶系数（`RiskAversionCoef`）
* $\operatorname{TC}$ 是交易成本惩罚函数
* $\lambda_1\ge 0$ 是买卖交易费率（`TurnoverPenaltyCoef`）
* $\lambda_2\ge 0$ 是买入交易费率（`BuyPenaltyCoef`）
* $\lambda_3\ge 0$ 是卖出交易费率（`SellPenaltyCoef`）
* $\mathbf{w}_0$ 是当前持有的组合权重
* $\mathbf{w}_b$ 是基准组合权重（当 `Benchmark=True` 时）
* $\mathfrak{C}$ 是约束条件自由组合成的约束集

均值方差模型导出的优化问题通常是一个凸规划问题（如果含有非零权重数目约束条件，则为整数规划问题），在某些参数取值特殊的情况下可以退化为二次规划或者线性规划问题。

## MeanVarianceObjective API 参考

### 构造方法

```python
MeanVarianceObjective(
    mask: NDArray[np.bool],                              # 股票池，True 表示可选
    expected_return: Optional[NDArray[np.float64]]=None, # 预期收益
    p0: Optional[NDArray[np.float64]]=None,              # 初始投资组合
    bmk: Optional[NDArray[np.float64]]=None,             # 基准投资组合
    factor_cov: Optional[NDArray[np.float64]]=None,      # 因子协方差阵 (k×k)
    factor_data: Optional[NDArray[np.float64]]=None,     # 因子暴露矩阵 (n×k)
    specific_risk: Optional[NDArray[np.float64]]=None,   # 特异性风险 (n,)
    cov: Optional[NDArray[np.float64]]=None,             # 证券协方差阵 (n×n)
    args: dict={},                                        # 参数设置
    config_file: Optional[str]=None                       # 配置文件路径
)
```

### 参数（通过 `args` 传入）

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Benchmark` | `bool` | `False` | 是否相对基准优化 |
| `ExpectedReturnCoef` | `float` | `0.0` | 收益项系数 $\gamma$ |
| `RiskAversionCoef` | `float` | `1.0` | 风险厌恶系数 $\lambda$ |
| `TurnoverPenaltyCoef` | `float` | `0.0` | 双向换手惩罚系数 $\lambda_1$ |
| `BuyPenaltyCoef` | `float` | `0.0` | 买入惩罚系数 $\lambda_2$ |
| `SellPenaltyCoef` | `float` | `0.0` | 卖出惩罚系数 $\lambda_3$ |

### 输入验证规则

- `Benchmark=True` 时 `bmk` 不能为 None
- `ExpectedReturnCoef != 0` 时 `expected_return` 不能为 None
- `RiskAversionCoef != 0` 时需要提供协方差信息（`cov` 或 `factor_cov + factor_data + specific_risk`）
- 有换手惩罚系数时 `p0` 不能为 None

## 示例：最小方差组合

当 $\gamma = 0$（`ExpectedReturnCoef=0`）时，模型退化为纯风险最小化——**最小方差组合**。

In [ ]:
import numpy as np
import cvxpy as cvx

# DEMO 数据
np.random.seed(0)
nID = 10
Mask = np.full(shape=(nID,), fill_value=True, dtype=np.bool)
ExpectedReturn = np.random.randn(nID)
Cov = np.cov(np.random.randn(100 * nID, nID), rowvar=False)
P0 = np.random.rand(nID)
P0 = P0 / np.sum(P0)
Bmk = np.random.rand(nID)
Bmk = Bmk / np.sum(Bmk)

In [ ]:
from QuantStudio.PortfolioConstructor.CVXPC import CVXPC
from QuantStudio.PortfolioConstructor.BasePC import MeanVarianceObjective, BudgetConstraint, WeightConstraint

Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, p0=P0, cov=Cov,
    args={"ExpectedReturnCoef": 0, "RiskAversionCoef": 1}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.SCIP, "verbose": True}})
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"组合波动率: {np.sqrt(Portfolio @ Cov @ Portfolio):.4f}")
print(f"求解状态: {Info['msg']}")

## 示例：均值方差组合（含预期收益）

设置 $\gamma > 0$ 来权衡收益与风险。下面的例子中 $\gamma=1, \lambda=2$，表示在最大化收益和最小化风险之间寻求平衡。

In [ ]:
Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, p0=P0, cov=Cov,
    args={"ExpectedReturnCoef": 1, "RiskAversionCoef": 2}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"组合预期收益: {Portfolio @ ExpectedReturn:.4f}")
print(f"组合波动率: {np.sqrt(Portfolio @ Cov @ Portfolio):.4f}")
print(f"求解状态: {Info['msg']}")

## 示例：相对基准的均值方差

设置 `Benchmark=True` 时，优化变量变为 $\mathbf{w} - \mathbf{w}_b$（相对基准的偏离权重），适用于指数增强策略。

In [ ]:
Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, p0=P0, bmk=Bmk, cov=Cov,
    args={"ExpectedReturnCoef": 1, "RiskAversionCoef": 5, "Benchmark": True}
)
ConstraintList = [
    WeightConstraint(mask=Mask, bmk=Bmk, up_limit=0.05, down_limit=-0.05, args={"Benchmark": True}),
    BudgetConstraint(mask=Mask, bmk=Bmk, args={"UpLimit": 0, "DownLimit": 0, "Benchmark": True})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
active_weight = Portfolio - Bmk
print(f"主动权重和: {np.sum(active_weight):.6f}")
print(f"跟踪误差: {np.sqrt(active_weight @ Cov @ active_weight):.4f}")
print(f"求解状态: {Info['msg']}")

## 示例：含交易成本的均值方差

通过换手惩罚系数控制调仓成本。下面的例子设置了 1% 的双边换手惩罚，使优化器倾向于减少不必要的调仓。

In [ ]:
Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, p0=P0, cov=Cov,
    args={
        "ExpectedReturnCoef": 1, "RiskAversionCoef": 2,
        "TurnoverPenaltyCoef": 0.01,  # 双边换手 1% 惩罚
    }
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"换手率: {np.sum(np.abs(Portfolio - P0)):.4f}")
print(f"求解状态: {Info['msg']}")

## 示例：使用因子模型输入协方差

当协方差矩阵通过 Barra 等多因子风险模型估计时，可以使用 `factor_cov` + `factor_data` + `specific_risk` 的形式输入，避免直接构造大型协方差矩阵。

In [ ]:
# 模拟因子模型数据
k = 5  # 因子数量
factor_cov = np.cov(np.random.randn(100, k), rowvar=False)
factor_data = np.random.randn(nID, k)
specific_risk = np.abs(np.random.randn(nID)) * 0.1

Objective = MeanVarianceObjective(
    mask=Mask,
    factor_cov=factor_cov, factor_data=factor_data, specific_risk=specific_risk,
    args={"ExpectedReturnCoef": 0, "RiskAversionCoef": 1}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"求解状态: {Info['msg']}")

## 示例：带约束的均值方差

组合多种约束条件构建更实用的优化问题。下面的例子同时施加了权重上下限、预算约束、波动率约束和因子暴露约束。

In [ ]:
from QuantStudio.PortfolioConstructor.BasePC import (
    VolatilityConstraint, FactorExposeConstraint, ExpectedReturnConstraint
)

# 模拟因子暴露数据
nFactors = 3
FactorData = np.random.randn(nID, nFactors)

Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, cov=Cov,
    args={"ExpectedReturnCoef": 1, "RiskAversionCoef": 3}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),       # 纯多头
    WeightConstraint(mask=Mask, up_limit=0.15),                   # 个股权重上限 15%
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1}),  # 全额投资
    VolatilityConstraint(mask=Mask, cov=Cov, args={"UpLimit": 0.10}),   # 年化波动率 ≤ 10%
    ExpectedReturnConstraint(mask=Mask, expected_return=ExpectedReturn, args={"DownLimit": 0.0}),  # 预期收益 ≥ 0
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"组合预期收益: {Portfolio @ ExpectedReturn:.4f}")
print(f"组合波动率: {np.sqrt(Portfolio @ Cov @ Portfolio):.4f}")
print(f"求解状态: {Info['msg']}")

## 示例：带约束松弛的优化

当约束条件过于严格导致问题不可行时，CVXPC 支持按优先级自动松弛约束。约束的 `DropPriority` 越大越先被松弛。

In [ ]:
Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, cov=Cov,
    args={"ExpectedReturnCoef": 1, "RiskAversionCoef": 1}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1}),
    # 以下约束设置了松弛优先级，求解失败时按优先级从大到小依次松弛
    VolatilityConstraint(mask=Mask, cov=Cov,
                         args={"UpLimit": 0.05, "DropPriority": 2}),  # 优先松弛
    ExpectedReturnConstraint(mask=Mask, expected_return=ExpectedReturn,
                             args={"DownLimit": 0.5, "DropPriority": 1}),  # 次优先松弛
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"求解状态: {Info['msg']}")
print(f"被松弛的约束: {Info['ReleasedConstraint']}")

## 均值方差模型的局限性与改进

如果不加任何约束，原始均值方差模型存在以下问题：

1. **参数估计误差大**。Chopra 和 Ziemba（1993）的研究表明，在风险厌恶为 50 的情况下，均值估计误差带来的效用损失远远高于方差和协方差；风险厌恶水平越高，对均值的估计误差越敏感。
2. **结果对输入参数非常敏感**。Michaud（1989）发现，输入参数的较小改变可能使结果大相径庭。
3. **优化结果可能过于集中**。Broadie（1993）的测试表明，在约束条件欠缺时，均值方差模型的结果容易集中在少数证券甚至一个证券上。
4. **换手率高，交易成本大**。De Carvalho、Lu 和 Moulin（2012）比较了 6 个组合优化模型，结果表明均值方差模型换手率较高。
5. **容易得到极端的分配结果**。Best 和 Grauer（1991）的研究表明，均值方差模型容易算出极大或极小的权重。
6. **较差的样本外表现**。DeMiguel et al.（2009）的结果表明，由于估计误差的存在，基于历史数据的均值方差组合在样本外表现很难超越等权重组合。

### 改进方法

Behr、Guettler 和 Miebs（2013）从权重约束角度对最小方差组合进行了改进：通过最小化协方差矩阵的 MSE 得到权重上下限，将样本协方差向一个特定协方差压缩，再求解带权重约束的最小方差组合。结果发现：权重约束最小方差组合夏普比比等权重组合高 30%，比市值加权高 60%，换手率也更低。

在 QuantStudio 中，可以通过组合以下手段来缓解上述问题：
- 使用 `WeightConstraint` 设置权重上下限
- 使用 `TurnoverConstraint` 或换手惩罚系数控制调仓
- 使用 `FactorExposeConstraint` 实现风格中性
- 使用因子模型（而非样本协方差）估计协方差矩阵